<a href="https://colab.research.google.com/github/BKBKlaassen/Gr8_ModelsForLanguageProcessing_assignments/blob/main/Group_08_M4LP_Assignment_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Assignment 7

You will explore typological parameters such as dominant word order and their encoding in the WALS database. You will also implement the BPE algorithm for tokenization and chrF for automatic evaluation of translation.

## 7.1 Cross-linguistic variation

For starters, we will look at the typological properties of languages, as defined e.g. in WALS (https://wals.info/).

Let's determine the word order parameters of Dutch.
For statistics of word order counts in Dutch, use the first 20 (complete) instances of each construction type from the OPUS corpus:
https://data.statmt.org/opus-100-corpus/v1.0/supervised/en-nl/opus.en-nl-dev.nl


7.1.1. What is the dominant order of Subject and Verb in Dutch? Is an alternative order possible? Confirm your answer by citing the frequency count among the first 20 instances of the relevant construction in the OPUS corpus. On each line, list an example preceded by a label (VS or SV). Each example should be a fragment of a sentence from OPUS with a subject noun or pronoun and a finite verb.

**Answer:** <details><summary>(Try to come to the answer yourself. Then you can click here to expand the solution)</summary>The dominant order is SV. Both SV and VS are possible but SV is a bit more frequent. This can be confirmed by corpus counts. 6:14 VS:SV

VS Naar mijn idee **was ze** hier elke avond, maar

SV **ik weet** niet zeker of

SV **ze hier was** die avond.

VS Hoever **gaat dat**?

SV **Wat hoort** daarin thuis?

SV **Die pillen hadden** nog een week moeten gaan.

SV (49) **De uitvoerprijs werd** vastgesteld op basis van de gegevens

SV **die verstrekt werden** door de Mitsui-groep en Eurostat.

SV **We zullen** binnenkort contact opnemen met de hoge autoriteiten

VS ... en miljoenen ongeboren kinderen in Europa **zou een strikt verbod** ...

SV **Ik zal** je even je kamer wijzen.

SV Want **Nederland had** een geweldig protectionistisch systeem, mevrouw.

SV **Ik weet** niet waar

SV **hij met zijn gedachten is**, maar

SV **het is** niet hier,

VS Hoe **kan deze informatie** worden verbeterd?

SV **Ik ben** meer mezelf dan ooit tevoren.

SV **Ik kan** niks.

VS **Kan ik** even terug en dat aannemen?

VS - Mijnheer de Voorzitter, ten eerste **wil ik** lof toezwaaien
</details>


7.1.2. (1 point) What is the dominant order of Genitive and Noun in Dutch? Genitive is any constituent expressing a possessor, e.g. _**my** mother_, _**John's** mother_, _the mother **of his friend**_. Confirm your answer by citing the frequency count among the first 20 instances of the relevant construction in the OPUS corpus. Count possessive pronouns (e.g. _mijn_, _jouw_) as instances of possessive constructions. On each line, list an example preceded by a label (GenN or NGen). Each example should be a fragment of a sentence from OPUS with a noun and a possessive, e.g. _wiens_, _Johans_, or _van Nederland_.

**Answer:**

7.1.3. (1 point) What is the dominant order of Adposition and Noun Phrase in Dutch? Confirm your answer by citing the frequency count among the first 20 instances of the relevant construction in the OPUS corpus.
On each line, list an example preceded by a label (AdpN or NAdp). Each example should be a fragment of a sentence from OPUS with a noun and a possessive, e.g. AdpN _Naar mijn idee_.

Does the other order exist in Dutch? If yes, give an example.


**Answer:**

7.1.4. You can confirm the word order parameters you determined for Dutch using WALS: the dominant order of Subject and the Verb, the dominant order of adposition and Noun Phrase, and the dominant order of genitive and noun. .

(2 points)

According to WALS, what other languages spoken in the European continent have the same combination of these parameters as Dutch?

**Answer:**

Cornish, Icelandic, Russian, German, French, Romanian, Albanian, Portuguese.

What differentiates closely related languages that don't have the same combination of parameters as Dutch?

**Answer**:  
Dutch has a Noun-Gentitive order, where English and Frisian have no-dominate Noun and Gentitive Order.

Now, with the help of WALS, find the northernmost language X spoken in South America (not the the northernmost in the world though!) that is opposite to Dutch with respect to parameters above: the dominant order of Subject and the Verb, the dominant order of adposition and Noun Phrase, and the dominant order of genitive and noun. What is the name of the language and in what countries is it spoken?

**Answer:**
Tirriyo predominantly Suriname and Brazil.


7.1.5 (1 point) Look up more information in WALS about the language X from the previous question. What is the order of Subject, Object, and Verb in X? Does language X have gender?

**Answer:**

7.1.6. How would the translation of the phrase "with rice" look in language X? Since you don’t know any words or morphemes in X, use English words in the grammatical structure of language X.

**Answer:** <details><summary>(click to see the answer)</summary>English _with rice_ translates into language X as _rice with_ because language X has noun-postposition word order</details>


7.1.7. (2 points) Given information in WALS, give an example of two sentences in English that will be translated as the same sentence in X. Since you don't know any words or morphemes in X, give a word by word translation of the two English sentences in language X.  Ignore articles in your X translation.

**Answer:**

7.1.8. (1 point) How would the translation of the sentence _The child did not sleep_ look in language X? Use information from WALS. Assume the most likely (=dominant) word order in your translation. Since you don’t know any words or morphemes in X, use a word by word translation in your answer. Ignore articles.

**Answer:**


## **7.2 Byte Pair Encoding**
In this section, we will implement the simplest version of Byte Pair encoding algorithm for training subword tokenization.

Download and uzip the English-Dutch parallel Europarl v7 corpus from https://opus.nlpl.eu/.

In [ ]:
!	curl -O https://object.pouta.csc.fi/OPUS-Europarl/v7/moses/en-nl.txt.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  204M  100  204M    0     0  24.2M      0  0:00:08  0:00:08 --:--:-- 31.0M


In [ ]:
!unzip en-nl.txt.zip

Archive:  en-nl.txt.zip
  inflating: Europarl.en-nl.en       
  inflating: Europarl.en-nl.nl       
  inflating: Europarl.en-nl.ids      
  inflating: README                  


Natural languages have open vocabularies with a long tail, i.e. there is a large number of very infrequent words. So in practice, it is not efficient (and often not realistic) to use every word as a processing unit whose properties have to be learned separately. Instead, less frequent words are broken into multiple subwords, usually via the byte pair encoding (BPE) algorithm. Now you will implement the BPE algorithm as it is introduced in SLP3, Chapter 2, pp.18-19.

7.2.1 (2 points) Define function `bigrams` that returns a list of bigrams in a list:

In [ ]:
def bigrams(lst):
  return[ f"{lst[i]}{lst[i+1]}" for i in range(0,len(lst)-1)]

You can check what your function does to the list of characters in the word _tree_:

In [ ]:
bigrams(['t','r','e','e','_'])

['tr', 're', 'ee', 'e_']

7.2.2 (2 points) Define function `replace_bp` that replaces each occurrence of the bigram `nextbp` with its concatenation. For example,

replace_bp(['t', 'r', 'e', 'e', '_'],'ee')

should give

['t','r', 'ee', '_']

In [ ]:
def replace_bp(y,nextbp):
  updated_char_list = []
  check = False
  for i in range(len(y)):
    if(check):
      check = False
      continue
    if(i+1 < len(y) and f"{y[i]}{y[i+1]}" == nextbp):
      updated_char_list.append(nextbp)
      check = True
    else:
      updated_char_list.append(y[i])
  return updated_char_list
replace_bp(['t', 'r', 'e', 'e', '_'],'ee')

['t', 'r', 'ee', '_']

7.2.3 (8 points) Train BPE vocabulary for subword tokenization

Now you can define a function bpe_train using replace_bp that reads a training corpus from file and returns a vocab of a certain size. Within the definition of _bpe_train_ function, make sure to insert an underscore '\_' symbol after each word (you can use the standard ```split()``` function to separate words from each other initially).

Use only the first 1000 lines of the corpus for BPE training - this suffices for our purposes

You can use a Counter object to count symbols or pairs of symbols (bigrams) and find the most frequent ones in the corpus.

In [ ]:
from collections import Counter

In [ ]:
def load_corpus(corpusfile, max_lines=1000):
  with open(corpusfile,'r') as src:
    z=src.readlines()[:max_lines]
  z=[x.strip().split() for x in z]
  return z

def bpe_train(corpus,targetbpe):
  raw_word_corpus = [list(word+"_") for sentence in load_corpus(corpus,1000) for word in sentence]
  vocab = set([token for word in raw_word_corpus for token in word])

  for i in range(targetbpe):
    current_bigram_counts = Counter([bigram for word in raw_word_corpus for bigram in bigrams(word)]) # generate bigram with occurence counter
    most_frequent_bigram = current_bigram_counts.most_common(1)[0][0] # find most frequent bigram
    vocab.add(most_frequent_bigram) # add to vocab
    raw_word_corpus = [replace_bp(word,most_frequent_bigram) for word in raw_word_corpus]
  return vocab


Train subword vocabularies of 1000 items on small Dutch and English corpora. You may want to debug your code on small target vocabularies first.

In [ ]:
nl_bpes=bpe_train("Europarl.en-nl.nl",1000)

['v', 'a', 'n', '_']
{'C', 'ä', 'ï', 'S', 'A', 'V', 'k', '2', 'ö', 'ó', 'L', '"', 'N', 'P', 'K', 'B', 's', 'c', 'á', ':', 'a', '?', 'à', 'D', '.', 'n', 'q', 'w', 'T', 'í', 'o', 'v', 'j', 'R', 'M', 'U', '5', '4', '9', 'H', 'l', 'ü', '7', 'O', '6', 'h', '_', "'", '8', 'X', 'è', '/', 'b', ';', 'g', 'J', 't', 'Z', ')', '-', 'ë', 'F', '!', 'm', 'u', 'f', '(', 'p', 'E', '%', 'x', ',', '0', 'i', '3', 'd', 'W', 'é', 'z', 'y', 'e', 'I', 'r', '1', 'G'}


In [ ]:
en_bpes=bpe_train("Europarl.en-nl.en",1000)

['o', 'f', '_']
{'C', 'ä', 'ï', 'S', 'A', 'V', 'k', '2', 'ó', 'L', '"', 'N', 'Q', 'P', 'K', 'B', 's', 'c', 'á', ':', 'a', '?', 'D', '.', 'n', '[', 'q', 'w', 'T', 'í', 'o', 'v', 'j', 'R', 'M', 'U', '5', '4', '9', 'H', 'l', 'ü', '7', 'O', '6', 'h', ']', '_', "'", '8', 'X', '/', 'b', ';', 'g', 'J', 't', ')', '-', 'Z', 'F', 'm', 'u', 'f', '(', 'p', 'E', 'Y', '%', 'x', ',', '0', 'i', '3', 'd', 'º', 'W', 'é', 'z', 'y', 'e', 'I', 'r', '1', 'G'}


What is the resulting list of subword units?

In [ ]:
nl_bpes

{'ion',
 'ateg',
 'we',
 'Ik_',
 'peri',
 'zek',
 'tegen_',
 'kle',
 'vra',
 'ale_',
 '1999_',
 'nieuw',
 'nation',
 'dat_',
 'arlemen',
 'vrouw_',
 'vraag',
 'vro',
 'idst',
 'oe',
 'beste',
 'hesie',
 '"',
 'atie_',
 'land',
 'richtsno',
 'alleen_',
 'un',
 'bij',
 'zen_',
 'P',
 'di',
 'moe',
 'heid_',
 'aar',
 'versch',
 'lo',
 'open_',
 'hou',
 'gens_',
 'a',
 '?',
 'programma_',
 'D',
 'nie',
 'veilig',
 'ol',
 'kan_',
 'binnen_',
 'die_',
 'moeten_',
 'aanda',
 'ver',
 'orm',
 'ist_',
 'no',
 'ontwikkel',
 'ang_',
 'uur_',
 'her',
 'behand',
 'is,_',
 'nat',
 'ef',
 'ze_',
 'ho',
 "collega's,_",
 'ste_',
 'wijze_',
 'ingen_',
 'veil',
 'wil',
 "regio'_",
 'w',
 'iteit_',
 'zien_',
 'ijf',
 'T',
 've_',
 'amendementen_',
 'ming_',
 'doelstell',
 "regio'",
 'í',
 'uitvoer',
 'geme',
 'in_',
 'ram',
 'v',
 'ie._',
 'ni',
 'nationale_',
 'd,_',
 'hebben_',
 'sit',
 'Commiss',
 'Al',
 'kun',
 'bur',
 'zelf',
 'ect',
 '5',
 '200',
 '9',
 'er_',
 'immers_',
 'sle',
 'commiss',
 'bev',


In [ ]:
en_bpes

{'ion',
 'So',
 'ment._',
 'say_',
 'cl',
 'therefore_',
 'them_',
 'ateg',
 'ans',
 'we',
 'inci',
 'ind',
 'sa',
 'ural_',
 'ion,_',
 'than_',
 'accoun',
 'ies,_',
 'up_',
 'gre',
 'e._',
 'necess',
 'have_',
 'Union_',
 'invol',
 'report_',
 'ement_',
 'where_',
 'tly_',
 'ial_',
 '"',
 'poin',
 'ud',
 'ment_',
 'important_',
 'uc',
 'appro',
 'ves_',
 'gh',
 'un',
 'ust_',
 'P',
 'di',
 'remo',
 'can_',
 'Hou',
 'pare',
 'vie',
 'Sch',
 'regard_',
 'lo',
 'proced',
 'our_',
 'a',
 '?',
 'the',
 'est',
 'ative_',
 'D',
 'objec',
 'ver',
 'aid_',
 'no',
 'once_',
 'ner',
 'account_',
 'pe',
 'ess',
 'Objec',
 'discus',
 'ved_',
 'nat',
 'into_',
 'is,_',
 'partic',
 'ef',
 'se',
 'ho',
 'other_',
 'princi',
 'mb',
 'There_',
 'ure._',
 'sib',
 'Committee_',
 'w',
 'particular_',
 'T',
 've_',
 'ates_',
 'uch_',
 'í',
 'ach',
 'in_',
 'ram',
 'v',
 'ance_',
 'States_',
 'd,_',
 'examp',
 'sit',
 'Commiss',
 'uct',
 'ect',
 '5',
 '200',
 '9',
 'er_',
 'his_',
 'competitiv',
 'low',
 'b

7.2.4 (4 points) Now define function bpe_tokenize that breaks down an input word (a string) into subwords in a greedy left to right manner using a subword vocabulary. *Greedy* tokenization means that whenever there are multiple options to identify the next syte pair unit, the longest possible one is selected.

In [ ]:
from IPython.core.pylabtools import retina_figure
def bpe_tokenize(s,bpes):
  relevent_entries = [entry for entry in bpes if s.__contains__(entry)]
  tokenized_word = []
  letter_pointer = 0
  while(letter_pointer < len(s)):
    current_longest = ''
    for entry in relevent_entries:
      for i in range(len(entry)):
        if s[letter_pointer+i] != entry[i]:
          break
        if len(current_longest) < len(entry) and i == len(entry)-1:
          current_longest = entry
    letter_pointer += len(current_longest)
    tokenized_word.append(current_longest)
  return tokenized_word


Test how your tokenizer works:

In [ ]:
bpe_tokenize('institution_',en_bpes)

['ion', 'stit', 'on', 'on_', 'u', 'ion_', 'ut', 'n_', 'it', 'io', 'ti', '_', 't', 'in', 's', 'stitut', 'n', 'o', 'st', 'i']


['in', 'stitut', 'ion_']

In [ ]:
bpe_tokenize('Europe_',en_bpes)

['pe', 'p', 'r', 'uro', 'Europe', 'e_', 'ro', 'pe_', 'u', 'ur', 'Europe_', 'op', '_', 'E', 'o', 'Euro', 'e']


['Europe_']

In [ ]:
bpe_tokenize('Europe_',nl_bpes)

['Europ', 'p', 'r', 'Eur', 'Europe', 'e_', 'ro', 'u', 'ur', 'op', '_', 'E', 'o', 'e']


['Europe', '_']

In [ ]:
bpe_tokenize('Europa_',nl_bpes)

['a', 'Europ', 'Europa_', 'a_', 'p', 'r', 'Eur', 'pa', 'ro', 'u', 'ur', 'op', '_', 'E', 'o']


['Europa_']

In [ ]:
bpe_tokenize('Nederland_',nl_bpes)

['land', 'a', 'la', 'r', 'N', 'der', 'de', 'an', 'l', '_', 'd', 'd_', 'er', 'and', 'n', 'and_', 'e']


['N', 'e', 'der', 'land', '_']

##7.3 Parallel corpora and automatic translation evaluation

**Exercise** (1 point) Look into the file `Europarl.en-nl.ids`. It contains four columns. What information do the 3rd and 4th columns encode? Name the task that has been performed here and explain it on the example of the third line of `Europarl.en-nl.ids`.

**Answer**

**Exercise: use Google Translate and process its output** (0.5 point) Take the first 10 lines of the Dutch corpus and translate it into English using Google Translate.

In [ ]:
#Paste the translation from Google translate below without any manual editing
google_translation = ''''''

Process the `google_translation` to get a list of strings each corresponding to one line of the original Dutch parallel data.

In [ ]:
translated_sentences=

**Exercise** (0.5 point) Read the first 10 lines from `Europarl.en-nl.en` as reference sentences.

In [ ]:
#your code
reference_sentences =

**Exercise** (3 points) Now implement chrF function for comparing a candidate translation with a reference translation, as discussed in the lecture.

In [ ]:
from collections import Counter
import re

def get_char_ngrams(text, n):
    #returns the Counter of all ngrams of length n in text, skipping spaces
    text = re.sub(r"\s+", "", text)
    #complete the function

def chrf(reference, candidate, max_n=2, beta=2):
  #your implementation will calculate f-scores for different ngram lengths up to max_n,
  #then return their average

Now we can evaluate the translations using chrF.

In [ ]:
cfrfs=[chrf(reference, candidate) for reference, candidate in zip(reference_sentences, translated_sentences)]
average_chrf = sum(cfrfs) / len(cfrfs)
print(f"Average chrF score: {average_chrf}")

How does this compare with an off-the-shelf implementation of chrF? NLTK contains a number of tools for machine translation, including alignment and automatic evaluation metrics. Among them, there is a chrF implementation:

In [ ]:
from nltk.translate.chrf_score import sentence_chrf

In [ ]:
cfrfs=[sentence_chrf(reference, candidate) for reference, candidate in zip(reference_sentences, translated_sentences)]
average_chrf = sum(cfrfs) / len(cfrfs)
print(f"Average NLTK chrF score: {average_chrf}")

**Exercise** (1 point) You will notice that the score is different from what your implementation computed. While small implementation details might differ, you can get it close to NLTK-produced with different parameter settings. Compute and print out the chrF score calculated with your implementation but with different parameters that closely corresponds to the NLTK implementation. Aim at less than 0.01 difference. Hint: look up NLTK documentation.

In [ ]:
#your code here

Congratulations! You have completed Assignment 7.


**The End.**